<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/riesgo/notebooks/c4_l4.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C4-L4 · VaR
VaR 95% histórico vs paramétrico y Expected Shortfall sobre 50 retornos y 20.000 de capital.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/riesgo/data/c4_l4.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c4_l4.csv'), Path('data/c4_l4.csv'), Path('c4_l4.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)

In [ ]:
var_hist = df['retorno'].quantile(0.05)
mu, sigma = df['retorno'].mean(), df['retorno'].std()
var_param = mu - 1.645 * sigma
es = df.loc[df['retorno'] <= var_hist, 'retorno'].mean()
print(f'mu={mu:+.5f}  sigma={sigma:.5f}')
print(f"VaR hist 95%={var_hist:.4f} ({var_hist*20000:+.0f} USD)  VaR param={var_param:.4f} ({var_param*20000:+.0f} USD)")
print(f"Expected Shortfall={es:.4f} ({es*20000:+.0f} USD)")

In [ ]:
import math
n = len(df)
cola = df.loc[df['retorno'] <= var_hist]
print(f'dias en cola: {len(cola)}/{n} ({len(cola)/n:.1%})')
print(df.nsmallest(5, 'retorno')[['dia','retorno','exposicion_neta']].to_string(index=False))
print(f"exposicion media={df['exposicion_neta'].mean():.3f}: a menor exposicion, menor VaR en USD")

In [ ]:
# Chequeo automático
assert var_hist < 0 and var_param < 0, 'el VaR de retornos debe ser negativo'
assert es <= var_hist, 'el Shortfall debe ser peor o igual que el VaR'
assert len(df) == 50
print('OK: VaR y Shortfall verificados')